# Part 2: Traffic Estimation Using Sublinear Space

In this section, space-efficient algorithms are used to estimate network properties without storing full datasets.

(a) Estimating Unique IPs

Exact Count

In [29]:
exact_unique_ips = len(set(df['source']).union(set(df['destination'])))
print("Exact unique IPs:", exact_unique_ips)

Exact unique IPs: 34801


The exact number of unique IPs is computed using full memory by storing all distinct source and destination IP addresses.

HyperLogLog

In [57]:
import hashlib
import numpy as np

class HyperLogLog:
    def __init__(self, b=10):
        self.b = b
        self.m = 2**b
        self.registers = np.zeros(self.m, dtype=int)

    def hash(self, x):
        # Use full 128-bit hash
        return int(hashlib.md5(x.encode()).hexdigest(), 16)

    def add(self, x):
        h = self.hash(x)

        idx = h & (self.m - 1)           # lower b bits
        w = h >> self.b                 # remaining bits

        rank = self._rho(w)
        self.registers[idx] = max(self.registers[idx], rank)

    def _rho(self, w):
        # count leading zeros in remaining 128-b bits
        binary = bin(w)[2:].zfill(128 - self.b)
        return binary.find('1') + 1

    def estimate(self):
        alpha = 0.7213 / (1 + 1.079 / self.m)
        Z = 1 / np.sum(2.0 ** (-self.registers))
        E = alpha * self.m**2 * Z

        # small range correction
        if E <= 2.5 * self.m:
            V = np.count_nonzero(self.registers == 0)
            if V > 0:
                E = self.m * np.log(self.m / V)

        return E

Run HLL

In [58]:
hll = HyperLogLog(b=10)

for ip in df['source']:
    hll.add(ip)

for ip in df['destination']:
    hll.add(ip)

estimate = int(hll.estimate())

Exact: 34801
Estimate: 35724
Error %: 2.6522226372805378


Error Analysis

In [59]:
print("Exact:", exact_unique_ips)
print("Estimate:", estimate)
print("Error %:", abs(estimate - exact_unique_ips)/exact_unique_ips * 100)

Exact: 34801
Estimate: 35724
Error %: 2.6522226372805378


Memory Comparison

In [60]:
print("Exact storage (IPs):", exact_unique_ips)
print("HLL registers:", hll.m)

Exact storage (IPs): 34801
HLL registers: 1024


The exact method requires storing all unique IPs, whereas HyperLogLog uses a fixed number of registers, significantly reducing memory usage.

(b) Heavy Hitters — Count-Min Sketch

Exact Top Destinations

In [34]:
exact_counts = df['destination'].value_counts()
top_exact = exact_counts.head(10)

top_exact

,count
destination,
198.164.30.2,232409
192.168.5.122,199437
203.73.24.75,193200
125.6.164.51,106826
67.220.214.50,49298
202.210.143.140,36189
82.98.86.183,25214
95.211.98.12,25095
209.112.44.10,21824


Count-Min Sketch

In [35]:
class CountMinSketch:
    def __init__(self, w=2000, d=5):
        self.w = w
        self.d = d
        self.table = np.zeros((d, w))

    def hash(self, x, i):
        return hash((x, i)) % self.w

    def add(self, x):
        for i in range(self.d):
            self.table[i][self.hash(x, i)] += 1

    def estimate(self, x):
        return min(self.table[i][self.hash(x, i)] for i in range(self.d))

Run CMS

In [36]:
cms = CountMinSketch()

for ip in df['destination']:
    cms.add(ip)

In [61]:
import pandas as pd

comparison = pd.DataFrame({
    "Exact": exact_counts.head(10),
    "Estimate": [cms.estimate(ip) for ip in exact_counts.head(10).index]
})

comparison

,Exact,Estimate
destination,,
198.164.30.2,232409,232467.0
192.168.5.122,199437,199496.0
203.73.24.75,193200,193248.0
125.6.164.51,106826,106860.0
67.220.214.50,49298,49492.0
202.210.143.140,36189,36217.0
82.98.86.183,25214,25861.0
95.211.98.12,25095,25170.0
209.112.44.10,21824,21877.0


Approx Top

In [37]:
approx_counts = {}

for ip in df['destination'].unique():
    approx_counts[ip] = cms.estimate(ip)

approx_top = sorted(approx_counts.items(), key=lambda x: -x[1])[:10]

approx_top

[('198.164.30.2', np.float64(232467.0)),
 ('192.168.5.122', np.float64(199496.0)),
 ('203.73.24.75', np.float64(193248.0)),
 ('125.6.164.51', np.float64(106860.0)),
 ('67.220.214.50', np.float64(49492.0)),
 ('202.210.143.140', np.float64(36217.0)),
 ('82.98.86.183', np.float64(25861.0)),
 ('95.211.98.12', np.float64(25170.0)),
 ('209.112.44.10', np.float64(21877.0)),
 ('62.140.213.243', np.float64(20552.0))]

Compare

In [38]:
for ip, est in approx_top[:5]:
    print(ip, "Exact:", exact_counts[ip], "Estimate:", est)

198.164.30.2 Exact: 232409 Estimate: 232467.0
192.168.5.122 Exact: 199437 Estimate: 199496.0
203.73.24.75 Exact: 193200 Estimate: 193248.0
125.6.164.51 Exact: 106826 Estimate: 106860.0
67.220.214.50 Exact: 49298 Estimate: 49492.0


Count-Min Sketch approximates frequency counts using sublinear space. While it may overestimate counts due to hash collisions, the most frequent elements are correctly identified.

(c) Bloom Filter — Membership

Bloom Filter

In [39]:
class BloomFilter:
    def __init__(self, size=100000, k=3):
        self.size = size
        self.k = k
        self.bits = np.zeros(size)

    def hash(self, x, i):
        return hash((x, i)) % self.size

    def add(self, x):
        for i in range(self.k):
            self.bits[self.hash(x, i)] = 1

    def check(self, x):
        return all(self.bits[self.hash(x, i)] for i in range(self.k))

Insert IPs

In [40]:
bf = BloomFilter()

for ip in df['source']:
    bf.add(ip)

False Positives

In [41]:
test_ips = df['destination'].sample(100)

false_positives = 0

for ip in test_ips:
    if bf.check(ip) and ip not in set(df['source']):
        false_positives += 1

print("False positives:", false_positives)

False positives: 1


Bloom filters provide efficient membership testing with a trade-off of false positives. The observed false positives demonstrate the probabilistic nature of the structure.